# Data Exploration — Arabic GSM8K v2

This notebook explores the **Omartificial-Intelligence-Space/Arabic-gsm8k-v2** dataset
used for training our Arabic Math Reasoning LLM.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

DATA_DIR = Path('../data')

## 1. Load Data

In [ ]:
pretrain_path = DATA_DIR / 'pretrain' / 'data.txt'
sft_path = DATA_DIR / 'finetune' / 'sft_data.json'

if pretrain_path.exists():
    pretrain_text = pretrain_path.read_text(encoding='utf-8')
    print(f'Pretraining text: {len(pretrain_text):,} chars, {len(pretrain_text.split()):,} words')
    print(f'Lines: {len(pretrain_text.splitlines()):,}')
    print(f'Size: {pretrain_path.stat().st_size / 1024:.1f} KB')
    print(f'\nFirst 500 chars:\n{pretrain_text[:500]}')
else:
    print('Run prepare_data.py first to download the dataset')

if sft_path.exists():
    with open(sft_path, 'r', encoding='utf-8') as f:
        sft_data = json.load(f)
    print(f'\nSFT pairs: {len(sft_data):,}')
    print(f'\nExample:')
    print(json.dumps(sft_data[0], ensure_ascii=False, indent=2))

## 2. Text Statistics

In [ ]:
if sft_path.exists():
    q_lengths = [len(s['input'].split()) for s in sft_data]
    a_lengths = [len(s['output'].split()) for s in sft_data]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(q_lengths, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
    axes[0].set_title('Question Length Distribution (words)')
    axes[0].set_xlabel('Words')
    axes[0].set_ylabel('Count')

    axes[1].hist(a_lengths, bins=30, color='coral', alpha=0.7, edgecolor='black')
    axes[1].set_title('Answer Length Distribution (words)')
    axes[1].set_xlabel('Words')
    axes[1].set_ylabel('Count')
    plt.tight_layout()
    plt.savefig('../results/plots/data_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Question lengths: mean={np.mean(q_lengths):.0f}, median={np.median(q_lengths):.0f}')
    print(f'Answer lengths: mean={np.mean(a_lengths):.0f}, median={np.median(a_lengths):.0f}')

## 3. Character & Word Frequency

In [ ]:
if pretrain_path.exists():
    char_freq = Counter(pretrain_text)
    word_freq = Counter(pretrain_text.split())

    print('Top 20 characters:')
    for ch, cnt in char_freq.most_common(20):
        display_ch = repr(ch) if ch.isspace() else ch
        print(f'  {display_ch}: {cnt:,}')

    print(f'\nUnique characters: {len(char_freq)}')
    print(f'Unique words: {len(word_freq)}')

    top_words = word_freq.most_common(20)
    fig, ax = plt.subplots(figsize=(12, 5))
    words, counts = zip(*top_words)
    ax.barh(range(len(words)), counts, color='teal')
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words)
    ax.invert_yaxis()
    ax.set_title('Top 20 Most Frequent Words')
    ax.set_xlabel('Frequency')
    plt.tight_layout()
    plt.savefig('../results/plots/word_frequency.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Tokenizer Analysis

In [ ]:
from src.tokenizer.bpe_tokenizer import BPETokenizer

tok_path = DATA_DIR / 'tokenizer.json'
if tok_path.exists():
    tokenizer = BPETokenizer.load(str(tok_path))
    print(f'Vocabulary size: {tokenizer.vocab_size}')
    print(f'BPE merges: {len(tokenizer.merges)}')

    test_sentences = [
        'كم عدد التفاحات المتبقية إذا كان لديك عشرة وأعطيت ثلاثة؟',
        'اشترى محمد ٣ أقلام بسعر ٥ ريالات',
        'ما هو حاصل ضرب ٧ في ٨؟',
    ]

    for sent in test_sentences:
        ids = tokenizer.encode(sent)
        decoded = tokenizer.decode(ids)
        details = tokenizer.get_token_details(sent)
        print(f'\nOriginal: {sent}')
        print(f'Tokens ({len(ids)}): {[d["token"] for d in details]}')
        print(f'IDs: {ids}')
        print(f'Decoded: {decoded}')
else:
    print('Tokenizer not found. Run prepare_data.py first.')